# 第四课：训练阶段与推理时计算

本 Notebook 集中放置理论、可编辑代码和验证练习。按顺序阅读，先预测，再运行，再修改代码解释结果。

## 1. 训练与推理

训练根据数据和目标更新模型参数；推理用已有参数完成当前任务，通常不更新参数。上下文变化不等于参数变化。

| 方法 | 常见材料或信号 | 目标 | 更新参数 |
| --- | --- | --- | --- |
| 预训练 | 大量文本序列 | 预测下一 token，形成语言与知识表示 | 是 |
| SFT | 问题与示范回答 | 提高示范回答的概率，学习回答方式 | 是 |
| 经典 RLHF | 人类偏好与奖励信号 | 先训练奖励模型，再通过强化学习更新策略 | 是 |
| DPO | 同一问题的偏好与非偏好回答对 | 相对参考策略直接优化偏好对 | 是 |
| 推理时计算 | 推理 token、候选、搜索或检查 | 用既定参数完成当前任务 | 通常否 |

这些方法不是每个模型必须依次经过的固定流水线。DPO 无需显式训练奖励模型并运行 RL 循环，也不是删除差回答后做 SFT。


## 2. 示范数据与偏好数据

SFT 提供“怎样回答”的示范；偏好数据提供同一问题下“哪个回答更好”的比较。偏好数据既可用于 DPO，也可用于经典 RLHF 中的奖励模型训练，不能仅凭数据形式确定算法。

下面只展示数据结构，不执行训练，因此运行这些单元格不会更新模型参数。


In [ ]:
sft_example = {"prompt": "7 + 11 等于多少？", "response": "18。"}
preference_example = {
    "prompt": "7 + 11 等于多少？",
    "chosen": "18。",
    "rejected": "17。",
}
print("SFT 示例：", sft_example)
print("偏好示例：", preference_example)


## 3. 候选 → 验证 → 选择

本实验不调用大模型，也不实现 SFT、RLHF 或 DPO。候选由我们预先写好，检查器将答案转换为整数，并与精确计算结果比较。

**运行前预测：** `'17'`、`'18'`、`'十八'`、`'无法判断'` 分别能否通过？


In [ ]:
candidates = ['17', '18', '十八', '无法判断']
expected = 7 + 11

def verify(answer):
    try:
        return int(answer.strip()) == expected
    except ValueError:
        return False

for answer in candidates:
    print(repr(answer), '检查通过：', verify(answer))
selected = next((answer for answer in candidates if verify(answer)), None)
print('选中的候选：', selected)


### 观察与解释

`int('18')` 得到整数 18，因此通过。`int('十八')` 抛出 ValueError，因此被拒绝，尽管语义正确。验收规则会影响结果。

**动手修改：** 在上一个代码单元中加入 `' 18 '`、`'18.0'`、`'答案是18'`，先预测结果，再重新运行。注意：修改候选列表后，需要重新运行选择逻辑。

## 4. 检查器出错会怎样？

下面故意把检查目标改成 17。先预测：即使正确答案在候选里，最终会选什么？


In [ ]:
wrong_target = 17  # 故意错误的验收目标

def faulty_verify(answer):
    try:
        return int(answer.strip()) == wrong_target
    except ValueError:
        return False

wrong_selection = next((a for a in candidates if faulty_verify(a)), None)
print('错误检查器选中：', wrong_selection)
print('选中答案通过正确检查器：', verify(wrong_selection) if wrong_selection is not None else False)


## 5. 自己调试与验证

依次完成，每次只改一个条件：

1. 删除候选中的 `'18'`，观察没有答案通过时返回什么。
2. 调换候选顺序，解释“选择第一个通过者”的含义。
3. 修改下面的检查器，使它也接受 `'十八'`。重新执行下方单元格。

扩展格式支持并不意味着解决了所有自然语言验证问题。简单算术可精确检验，开放式回答的检查器可能误判。更多推理时计算不保证每道题更好，还需测量正确率、成本和时延；输出解释更长也不代表内部推理更好。


In [ ]:
def my_verify(answer):
    # TODO：先处理中文“十八”，再尝试转换整数。
    try:
        return int(answer.strip()) == expected
    except ValueError:
        return False

for answer in ['18', '十八', '17', '无法判断']:
    print(repr(answer), my_verify(answer))
# 修改完成后，预期依次为 True、True、False、False。


## 6. 与 Agent 的联系

模型生成工具调用 → 外部软件执行工具 → 工具结果加入上下文 → 模型继续生成。上下文变了，但通常没有更新参数；工具循环是外部软件逻辑，底层生成仍然是自回归。

## 7. 用自己的话验收

1. 预训练与 SFT 的常见训练材料有什么区别？
2. SFT 与偏好优化的数据有什么区别？
3. 经典 RLHF 与 DPO 的流程有什么区别？
4. “训练更多”和“回答时算得更多”有什么区别？
5. 为什么正确候选存在，也可能选错答案？

### 我的预测、观察与解释

- 预测：
- 实际观察：
- 原因：
- 还不清楚的问题：

课堂记录（2026-10-10）：已通过 SFT、DPO 与推理时计算的分步识别；已理解错误检查器可能选错答案。预训练与经典 RLHF 尚待独立验收。以上代码仍建议自行逐格运行与修改。

## 参考原始资料

- [InstructGPT：SFT、奖励模型与 RLHF](https://arxiv.org/abs/2203.02155)
- [DPO 原论文](https://arxiv.org/abs/2305.18290)
- [推理训练与推理时计算示例](https://openai.com/index/learning-to-reason-with-llms/)
